<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/05_WOA_Feature_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")

import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here

RESULTS_PATH = get_results_path(PROJECT_PATH)

df = load_dataset(PROJECT_PATH)

X, y, groups = split_xy(df)

X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)
X_train, X_test = drop_near_constant(X_train, X_test)

n_features = X_train.shape[1]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Number of features:", n_features)

Constant features removed: 35
Training shape: (603, 718)
Testing shape: (153, 718)
Number of features: 718


In [4]:
classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

cv = make_cv(groups_train)

alpha = 0.9

print("Classifier and CV are ready.")

Classifier and CV are ready.


In [5]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))


def continuous_to_binary(position, rng):
    probabilities = sigmoid(position)

    random_values = rng.random(size=position.shape)

    binary_position = (random_values < probabilities).astype(int)

    return binary_position

def calculate_coefficients(iteration, max_iterations, rng):
    a = 2 - iteration * (2 / max_iterations)

    r1 = rng.random()
    r2 = rng.random()

    A = 2 * a * r1 - a
    C = 2 * r2

    return A, C

def update_whale_positions(
    positions,
    best_position,
    iteration,
    max_iterations,
    rng,
    lower_bound=-6,
    upper_bound=6
):
    new_positions = np.zeros_like(positions)

    a = 2 - iteration * (2 / max_iterations)

    n_whales = positions.shape[0]

    for i in range(n_whales):

        X = positions[i].copy()

        r1 = rng.random()
        r2 = rng.random()

        A = 2 * a * r1 - a
        C = 2 * r2

        p = rng.random()

        if p < 0.5:

            if abs(A) < 1:

                # Encircling the best solution
                D = np.abs(C * best_position - X)
                new_position = best_position - A * D

            else:

                # Search for a random whale
                random_index = rng.integers(0, n_whales)
                random_position = positions[random_index]

                D = np.abs(C * random_position - X)
                new_position = random_position - A * D

        else:

            # Spiral movement
            b = 1
            l = rng.uniform(-1, 1)

            D_prime = np.abs(best_position - X)

            new_position = (
                D_prime * np.exp(b * l) * np.cos(2 * np.pi * l)
                + best_position
            )

        # Keep positions within the search boundaries
        new_positions[i] = np.clip(
            new_position,
            lower_bound,
            upper_bound
        )

    return new_positions

def population_to_binary(positions, rng):
    probabilities = sigmoid(positions)

    random_values = rng.random(size=positions.shape)

    binary_population = (
        random_values < probabilities
    ).astype(int)

    return binary_population

def fitness_function(
    binary_position,
    X_train,
    y_train,
    classifier,
    cv,
    alpha=0.9
):
    # Convert input to a NumPy array
    binary_position = np.asarray(binary_position)

    # Find selected feature indices
    selected_indices = np.where(binary_position == 1)[0]

    # Count selected features
    n_selected = len(selected_indices)

    # Reject solutions with no selected features
    if n_selected == 0:
        return 1.0

    # Select only the chosen features
    X_selected = X_train.iloc[:, selected_indices]

    # Evaluate classification performance using cross-validation
    cv_scores = cross_val_score(
        classifier,
        X_selected,
        y_train,
        cv=cv,
        scoring="f1_macro"
    )

    mean_f1 = cv_scores.mean()

    # Calculate feature selection ratio
    feature_ratio = n_selected / X_train.shape[1]

    # Calculate final fitness
    fitness = (
        alpha * (1 - mean_f1)
        + (1 - alpha) * feature_ratio
    )

    return fitness

def binary_woa(
    X_train,
    y_train,
    classifier,
    cv,
    population_size=10,
    max_iterations=20,
    alpha=0.9,
    lower_bound=-6,
    upper_bound=6,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    n_features = X_train.shape[1]

    # 1. Initialize continuous whale positions
    positions = rng.uniform(
        lower_bound,
        upper_bound,
        size=(population_size, n_features)
    )

    # 2. Convert initial population to binary
    binary_population = population_to_binary(
        positions,
        rng
    )

    # 3. Evaluate initial population
    fitness_values = np.array([
        fitness_function(
            binary_population[i],
            X_train,
            y_train,
            classifier,
            cv,
            alpha
        )
        for i in range(population_size)
    ])

    # 4. Find initial global best
    best_index = np.argmin(fitness_values)

    best_fitness = fitness_values[best_index]
    best_position = positions[best_index].copy()
    best_binary_solution = binary_population[best_index].copy()

    convergence_curve = [best_fitness]

    # 5. Main WOA iterations
    for iteration in range(max_iterations):

        # Update continuous positions
        positions = update_whale_positions(
            positions=positions,
            best_position=best_position,
            iteration=iteration,
            max_iterations=max_iterations,
            rng=rng,
            lower_bound=lower_bound,
            upper_bound=upper_bound
        )

        # Convert updated positions to binary
        binary_population = population_to_binary(
            positions,
            rng
        )

        # Evaluate new population
        fitness_values = np.array([
            fitness_function(
                binary_population[i],
                X_train,
                y_train,
                classifier,
                cv,
                alpha
            )
            for i in range(population_size)
        ])

        # Find best candidate in current iteration
        current_best_index = np.argmin(fitness_values)
        current_best_fitness = fitness_values[current_best_index]

        # Update global best if improvement is found
        if current_best_fitness < best_fitness:

            best_fitness = current_best_fitness

            best_position = positions[current_best_index].copy()

            best_binary_solution = (
                binary_population[current_best_index].copy()
            )

        # Store best fitness so far
        convergence_curve.append(best_fitness)

        print(
            f"Iteration {iteration + 1}/{max_iterations} "
            f"| Best Fitness: {best_fitness:.6f} "
            f"| Selected Features: {best_binary_solution.sum()}"
        )

    return (
        best_binary_solution,
        best_fitness,
        convergence_curve
    )

print("WOA functions are ready.")

WOA functions are ready.


In [6]:
population_size = WOA_SETTINGS[ACTIVE_DATASET]["population_size"]
max_iterations = WOA_SETTINGS[ACTIVE_DATASET]["max_iterations"]

random_seeds = WOA_SETTINGS[ACTIVE_DATASET]["random_seeds"]

print("Population size:", population_size)
print("Maximum iterations:", max_iterations)
print("Random seeds:", random_seeds)

Population size: 10
Maximum iterations: 20
Random seeds: [42, 100, 2026]


In [7]:
all_runs = []

for seed in random_seeds:

    print(f"\nStarting WOA run with seed {seed}")

    best_solution, best_fitness, convergence_curve = binary_woa(
        X_train=X_train,
        y_train=y_train,
        classifier=classifier,
        cv=cv,
        population_size=population_size,
        max_iterations=max_iterations,
        alpha=alpha,
        lower_bound=-6,
        upper_bound=6,
        random_state=seed
    )

    all_runs.append({
        "seed": seed,
        "best_solution": best_solution.copy(),
        "best_fitness": best_fitness,
        "selected_features": int(best_solution.sum()),
        "convergence_curve": convergence_curve.copy()
    })

print("\nAll WOA runs completed.")


Starting WOA run with seed 42
Iteration 1/20 | Best Fitness: 0.273514 | Selected Features: 111
Iteration 2/20 | Best Fitness: 0.273514 | Selected Features: 111
Iteration 3/20 | Best Fitness: 0.263177 | Selected Features: 32
Iteration 4/20 | Best Fitness: 0.257555 | Selected Features: 37
Iteration 5/20 | Best Fitness: 0.257555 | Selected Features: 37
Iteration 6/20 | Best Fitness: 0.243735 | Selected Features: 45
Iteration 7/20 | Best Fitness: 0.243735 | Selected Features: 45
Iteration 8/20 | Best Fitness: 0.243735 | Selected Features: 45
Iteration 9/20 | Best Fitness: 0.243735 | Selected Features: 45
Iteration 10/20 | Best Fitness: 0.239802 | Selected Features: 31
Iteration 11/20 | Best Fitness: 0.239802 | Selected Features: 31
Iteration 12/20 | Best Fitness: 0.239802 | Selected Features: 31
Iteration 13/20 | Best Fitness: 0.239802 | Selected Features: 31
Iteration 14/20 | Best Fitness: 0.239802 | Selected Features: 31
Iteration 15/20 | Best Fitness: 0.239802 | Selected Features: 31
I

In [8]:
runs_summary = pd.DataFrame([
    {
        "Seed": run["seed"],
        "Best Fitness": run["best_fitness"],
        "Selected Features": run["selected_features"]
    }
    for run in all_runs
])

runs_summary

,Seed,Best Fitness,Selected Features
0,42,0.239802,31
1,100,0.240141,120
2,2026,0.244903,101


In [9]:
best_run_index = np.argmin([
    run["best_fitness"] for run in all_runs
])

best_run = all_runs[best_run_index]

final_woa_solution = best_run["best_solution"].copy()
final_woa_fitness = best_run["best_fitness"]

print("Best seed:", best_run["seed"])
print("Best Fitness:", final_woa_fitness)
print("Selected features:", final_woa_solution.sum())

Best seed: 42
Best Fitness: 0.23980208611242493
Selected features: 31


In [10]:
final_selected_features = X_train.columns[
    final_woa_solution == 1
].tolist()

pd.DataFrame({
    "Feature": final_selected_features
}).to_csv(
    RESULTS_PATH / "final_woa_selected_features.csv",
    index=False
)

runs_summary.to_csv(
    RESULTS_PATH / "woa_multiple_runs_summary.csv",
    index=False
)

print("Best selected features saved.")
print("Multiple runs summary saved.")

Best selected features saved.
Multiple runs summary saved.
